# T1-01 EDA: 대상 통계표 프로파일

**이 노트북이 답하려는 질문:** 실제 KOSIS 표가 B0 챗봇 코드와 전처리 규칙(R1~R9)이 가정한 모양과 같은가?

- B0 코드는 지금까지 가짜 표로만 검증됐다. 실제 표 5개에서 어긋나는 곳을 찾아 다음 태스크(T1-02 질문 유형, T1-05 전처리, T1-03 골든셋, T1-08 거절)에 넘긴다.
- 데이터는 `notebooks/kosis_fetch_tables.ipynb`로 한 번 받아 둔 파일(`data/raw/kosis/`)만 읽는다. API는 부르지 않는다.
- 결과는 셀 값이 아니라 **구조와 개수**로 남긴다. 예: "결측 기호 x가 12개", "분류 3단계".

**진행 방식:** 아래 단위를 위에서부터 하나씩 실행한다. 단위마다 (1) 셀을 돌리고 (2) 셀 위 "읽는 법"으로 해석하고 (3) `docs/eda.md`에 한 절씩 쓴다.

| 단위 | 측정 | 넘기는 곳 |
| --- | --- | --- |
| 0 | 공통 준비: 설정, 메타 계층, 표 로드 | - |
| 1 | 입력 형식, 로더 통과 | T1-05 |
| 2 | 크기 | R7 |
| 3 | 분류 체계, 합계 정합성 | R1·R2·R8·R9 |
| 4 | 시점 | R3 |
| 5 | 단위, 측정 종류 | R5, T1-07 |
| 6 | 값·결측 기호, 빠진 조합 | R4, R6, T1-08 |
| 7 | 질문 가능성 매트릭스 | T1-02, T1-03 |
| 8 | 저장과 표 간 비교 | `docs/eda.md` |

자동 판정(비율 표 추정, 계 이름 추정, 질문 가능 여부)은 휴리스틱이다. 마지막에는 사람이 확인한다.

## 단위 0. 공통 준비

### 0-1. 설정

**이 셀이 하는 일:** 레포 루트를 찾아 `kosis_chat`(B0 패키지)을 import하고, 대상 표 5개와 경로, 공통 도우미 함수를 정한다.

- `TOTAL_NAMES`: 이름으로 "계"를 추정하는 패턴. 단위 3에서 메타 계층과 비교하려고 둔다.
- `RATIO_UNITS`: 단위 이름에 %, 율, 당, ‰가 있으면 비율 표로 추정한다(단위 5).
- `md_table`: `docs/eda.md`에 붙여 넣을 마크다운 표를 만든다.

다른 단위는 모두 이 셀과 0-2, 0-3을 먼저 실행해야 돌아간다.

In [ ]:
# Setup: repo root on sys.path, target tables, paths, shared helpers.
import json
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

root = Path.cwd()
while not (root / "kosis_chat").is_dir() and root != root.parent:
    root = root.parent
assert (root / "kosis_chat").is_dir(), "open this notebook from inside the repo"
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from kosis_chat.data.loader import load_table
from kosis_chat.data.preprocessing import clean
from kosis_chat.data.schema import SchemaError
from kosis_chat.data.serializer import to_markdown

TABLE_IDS = ["DT_1B8000G", "DT_1B26006_A01", "DT_1B83A35", "DT_1BPA001", "DT_1B81A24"]
RAW_DIR = root / "data" / "raw" / "kosis"
META_PATH = root / "data" / "통계표_메타데이터.jsonl"
OUT_DIR = root / "data" / "eda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOTAL_NAMES = re.compile(r"^(계|합계|총계|전체|전국|총인구|소계)$|^(총|전체)\s")  # name-based guess, compared with meta hierarchy
RATIO_UNITS = re.compile(r"%|퍼센트|율|비\b|당|‰")


def md_table(df):
    """Markdown table without the optional tabulate dependency."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


def show(rows):
    """Print a list of dicts as a markdown table."""
    print(md_table(pd.DataFrame(rows)) if rows else "(no rows)")


def decimals(raw):
    m = re.search(r"\.(\d+)", str(raw).replace(",", ""))
    return len(m.group(1)) if m else 0

### 0-2. 메타데이터의 분류 계층

**이 셀이 하는 일:** 공유받은 `data/통계표_메타데이터.jsonl`(2,364표)에서 대상 표 5개만 골라 읽고, 분류마다 코드별 계층 정보를 만든다.

- `hierarchy(meta)` → `{분류명: {코드: {name, lvl, up, leaf}}}`
  - `lvl`: 단계(1이 최상위), `up`: 상위 코드(`upItmId`), `leaf`: 맨 아래 범주인지
- B0에는 계·소계를 가리는 규칙(R2)이 아직 없어서, 이 계층을 단위 3(분류 체계, 합계 정합성)과 단위 7에서 쓴다.

**확인할 것:** `missing`이 비어 있어야 한다. 비어 있지 않으면 그 표는 계층 정보 없이 분석된다(단위 3 결과가 0으로 나온다).

In [ ]:
# Shared metadata for the target tables: classification hierarchy (lvl, upItmId, leaf) per category code.
meta_by_id = {}
with META_PATH.open(encoding="utf-8-sig") as f:
    for line in f:
        rec = json.loads(line)
        if rec.get("table_id") in TABLE_IDS:
            meta_by_id[rec["table_id"]] = rec
print("meta found:", sorted(meta_by_id), "missing:", sorted(set(TABLE_IDS) - set(meta_by_id)))


def hierarchy(meta):
    """{class name: {code: {'name', 'lvl', 'up', 'leaf'}}} from the shared metadata."""
    out = {}
    for cls in meta.get("classifications") or []:
        name = cls.get("classNm") or cls.get("classId")
        codes = {}
        for it in cls.get("itmList") or []:
            codes[str(it.get("itmId"))] = {
                "name": it.get("scrKor"),
                "lvl": int(it["lvl"]) if str(it.get("lvl") or "").isdigit() else None,
                "up": str(it.get("upItmId")) if it.get("upItmId") not in (None, "") else None,
                "leaf": str(it.get("leaf")).lower() in ("true", "y", "1") if it.get("leaf") is not None else None,
            }
        out[name] = codes
    return out


hier_by_id = {tid: hierarchy(m) for tid, m in meta_by_id.items()}

### 0-3. 받은 파일 읽기 (B0 로더 + 전처리)

**이 셀이 하는 일:** 표마다 `data/raw/kosis/` 아래 파일을 찾아 B0의 `clean(load_table(path))`를 그대로 돌리고, 결과를 DataFrame으로 바꿔 `loaded`에 담는다. 새 파서는 만들지 않는다.

- DataFrame 열: `item`, `period`, `value_raw`(원문), `value`(숫자), `flag`(value_flag), `unit`, `dim:<분류명>`(범주 이름), `code:<분류명>`(범주 코드)
- 표 하나가 주기별(Y/M/Q) 파일 여러 개일 수 있다. 분석은 **파일 단위**로 한다.
- 이후 단위들은 모두 `loaded`와 `results`를 쓴다. `results[(표ID, 파일명)]`에 단위별 결과가 쌓이고 단위 8에서 저장된다.

로더 통과 여부의 해석은 바로 아래 단위 1에서 한다.

In [ ]:
# Load every downloaded file of the target tables through the B0 loader and cleaner.
def table_files(tbl_id):
    return sorted(p for p in RAW_DIR.rglob(f"*_{tbl_id}_*.json") if not p.name.endswith(".meta.json"))


def to_frame(table):
    rows = []
    for c in table.cells:
        row = {"item": c.item, "period": c.period_raw, "value_raw": c.value_raw, "value": c.value,
               "flag": c.value_flag, "unit": c.unit}
        for k, v in c.dims.items():
            row[f"dim:{k}"] = v
        for k, v in c.dim_codes.items():
            row[f"code:{k}"] = v
        rows.append(row)
    return pd.DataFrame(rows)


loaded = {}    # (tbl_id, file name) -> (TableInput, DataFrame)
failures = {}  # (tbl_id, file name) -> loader problems
results = {}   # (tbl_id, file name) -> {unit name: measurements}
for tbl_id in TABLE_IDS:
    files = table_files(tbl_id)
    if not files:
        print(f"{tbl_id}: no downloaded file in {RAW_DIR.relative_to(root)} (run kosis_fetch_tables.ipynb first)")
    for path in files:
        key = (tbl_id, path.name)
        try:
            t = clean(load_table(path))
            loaded[key] = (t, to_frame(t))
            results[key] = {"tbl_nm": t.meta.tbl_nm, "survey": t.meta.survey_nm, "period_type": t.meta.period_type}
        except SchemaError as exc:
            failures[key] = [str(p) for p in exc.problems]
print(f"loaded {len(loaded)} files, failed {len(failures)}")

## 단위 1. 입력 형식·로더 통과

**알고 싶은 것:** 받은 파일을 우리 챗봇(B0 로더)이 그대로 읽을 수 있나?

**방법**
- 0-3에서 로더를 통과한 파일과 실패한 파일을 표로 본다.
- 원본 JSON 첫 행의 필드명(C1_OBJ_NM, ITM_NM, PRD_DE, DT 등)을 함께 출력해 B0 로더가 기대한 필드와 비교한다.

**읽는 법**
- 전부 OK면 입력 형식은 문제없다.
- FAIL이면 같이 나오는 problems가 곧 로더가 실제 응답에서 놓친 점이다(필드 이름 차이, 메타 누락 등). T1-05에서 고칠 1순위다.
- 원본 필드명 중 B0가 쓰지 않는 필드(예: C2~C8, 영문명)가 있으면 버려지는 정보가 무엇인지 적어 둔다.
- 표 하나가 주기별 파일 여러 개로 나온 것은 오류가 아니다(예: DT_1B8000G 월·분기·연간).

**docs/eda.md에 쓸 것:** 표별 파일 수, 통과/실패, 실패 사유 요약, 원본 필드 목록.

In [ ]:
# Unit 1: loader pass/fail per file and the raw field names the API returned.
rows = []
for tbl_id in TABLE_IDS:
    for path in table_files(tbl_id):
        key = (tbl_id, path.name)
        raw = json.loads(path.read_text(encoding="utf-8"))
        fields = sorted(raw[0]) if isinstance(raw, list) and raw else []
        status = "OK" if key in loaded else "FAIL"
        rows.append({"표": tbl_id, "파일": path.name, "로더": status,
                     "셀": len(loaded[key][0].cells) if key in loaded else "-",
                     "원본 필드": ", ".join(fields)})
        if key in results:
            results[key]["input"] = {"raw_fields": fields, "loader": status}
show(rows)
for (tbl_id, fname), problems in failures.items():
    print(f"\nFAIL {tbl_id}/{fname}:")
    for p in problems[:10]:
        print("  -", p)

## 단위 2. 크기·구조

**알고 싶은 것:** 표가 얼마나 크고, 프롬프트에 통째로 넣을 수 있나?

**방법**
- `cells`: 행 수, `items`: 항목 수, `codes_per_dim`: 분류별 범주 수
- `approx_tokens`: B0 `to_markdown`으로 직렬화한 글자 수 ÷ 2 (한국어 기준 대략치)

**읽는 법**
- 셀 수 ≈ 항목 × 범주 수들의 곱 × 시점 수다. 어느 분류가 크기를 키우는지 본다(시군구, 연령 등).
- `approx_tokens`가 수천을 넘으면 B0처럼 표를 통째로 넣는 방식은 실제 표에서 안 된다. R7(필요한 행만 고르기)이 필요하다는 근거다.
- 다단 분류 표(DT_1BPA001)와 시군구 표(DT_1B83A35)가 가장 클 것으로 예상하고 확인한다.

**docs/eda.md에 쓸 것:** 표별 셀·항목·분류(범주 수)·토큰 수 한 줄씩.

In [ ]:
# Unit 2: size of each file and its serialized prompt length.
rows = []
for key, (table, df) in loaded.items():
    dims = table.dim_names
    codes_per_dim = {d: df[f"code:{d}"].nunique() if f"code:{d}" in df else df[f"dim:{d}"].nunique() for d in dims}
    md_chars = len(to_markdown(table))
    results[key]["size"] = {"cells": len(df), "items": df["item"].nunique(), "dims": dims,
                            "codes_per_dim": codes_per_dim, "markdown_chars": md_chars, "approx_tokens": md_chars // 2}
    rows.append({"표": key[0], "파일": key[1], "셀": len(df), "항목": df["item"].nunique(),
                 "분류(범주 수)": " × ".join(f"{d}({n})" for d, n in codes_per_dim.items()) or "-",
                 "토큰(대략)": md_chars // 2})
show(rows)

## 단위 3. 분류 체계와 합계 정합성

두 측정은 같은 계층 정보(메타의 `lvl`, `upItmId`, `leaf`)를 쓰고, 둘 다 "계를 어떻게 다룰까(R2·R9)"에 답하므로 한 단위로 묶었다.

### 3-1. 분류 체계

**알고 싶은 것:** 분류 안에 계·소계·묶음 구간(예: 15~64세)이 어디에 있나?

**방법:** 데이터에 실제로 나온 코드를 0-2의 계층과 맞춘다.
- `codes_in_data` / `codes_in_meta` / `not_in_meta`: 데이터 코드 수, 메타 코드 수, 메타에 없는 데이터 코드 수
- `levels`: 단계별 코드 수, `non_leaf_in_data`: 데이터에 나온 상위 범주 수
- `roots`: 메타상 최상위 이름, `total_like_names`: 이름(계·합계·전국 등)으로 추정한 총계

**읽는 법**
- `non_leaf_in_data > 0`이면 상위와 하위가 한 표에 같이 있다. 그대로 다 더하면 이중 계산이다. R2를 이름이 아니라 계층으로 판정해야 한다는 근거다.
- `roots`와 `total_like_names`가 다르면 이름 기반 판정을 믿으면 안 된다는 증거다.
- `not_in_meta > 0`이면 메타에 없는 코드가 데이터에 있다. 그 분류는 메타를 그대로 믿을 수 없다.

In [ ]:
# Unit 3-1: classification hierarchy as seen in the data vs the shared metadata.
rows = []
for key, (table, df) in loaded.items():
    hier = hier_by_id.get(key[0], {})
    out = {}
    for d in table.dim_names:
        codes = hier.get(d) or {}
        seen = set(df[f"code:{d}"]) if f"code:{d}" in df else set()
        lvls = Counter(codes[c]["lvl"] for c in seen if c in codes)
        out[d] = {
            "codes_in_data": len(seen), "codes_in_meta": len(codes), "not_in_meta": len(seen - set(codes)),
            "levels": dict(sorted((k, v) for k, v in lvls.items() if k is not None)),
            "non_leaf_in_data": sum(1 for c in seen if c in codes and codes[c]["leaf"] is False),
            "roots": sorted({codes[c]["name"] for c in seen if c in codes and not codes[c]["up"]}),
            "total_like_names": sorted({codes[c]["name"] for c in seen
                                        if c in codes and TOTAL_NAMES.search(codes[c]["name"] or "")}),
        }
        rows.append({"표": key[0], "파일": key[1], "분류": d, **{k: v for k, v in out[d].items()}})
    results[key]["hierarchy"] = out
show(rows)

### 3-2. 합계 정합성

**알고 싶은 것:** 상위 범주 값 = 하위 범주 값의 합인가?

**방법:** `sum_check()`가 메타 계층으로 부모-자식 코드를 묶고, 다른 조건(항목, 시점, 다른 분류)을 고정한 채 `|부모 − Σ자식|`을 계산한다. 값이 ok·잠정인 셀만 쓰고, 비율 표(단위 이름에 %·율 등)는 건너뛴다.
- `checked`: 검사한 부모 수
- `exact`: 차이 0
- `within_rounding`: 차이 ≤ 0.5 × 자식 수 × 10^(−소수 자릿수). 자식 하나마다 반 단위까지 반올림됐을 수 있다는 뜻이다.
- `max_abs_diff`: 가장 큰 차이

**읽는 법**
- `within_rounding / checked`가 거의 1이면 표의 계를 그대로 써도 안전하다(R9 분모 규칙).
- 크게 어긋나면 비공개 셀이 빠졌거나 계층 해석이 틀린 것이다. 그 표는 직접 합산하지 말고 표의 계만 쓴다.
- `checked = 0`은 통과가 아니다. 계층 정보가 부족해서 판단 못 한 것이다.

**docs/eda.md에 쓸 것 (3-1, 3-2):** 분류별 단계 수·상위 범주 유무·이름 추정과 계층의 차이, 합계 일치 `정확/반올림 내/검사`.

In [ ]:
# Unit 3-2: parent value vs sum of its children (count-like units only).
def is_ratio(df):
    return any(RATIO_UNITS.search(u or "") for u in set(df["unit"]))


def sum_check(df, dims, hier):
    """Parent value vs sum of its child codes, other dims/item/period fixed."""
    checked = exact = near = 0
    worst = 0.0
    ok = df[df["flag"].isin(["ok", "provisional"])]
    for dim in dims:
        codes = hier.get(dim) or {}
        children = defaultdict(list)
        for code, info in codes.items():
            if info["up"] and info["up"] in codes:
                children[info["up"]].append(code)
        if not children or f"code:{dim}" not in ok:
            continue
        keys = [c for c in ok.columns if c.startswith("code:") and c != f"code:{dim}"] + ["item", "period"]
        for _, grp in ok.groupby(keys, dropna=False):
            vals = dict(zip(grp[f"code:{dim}"], grp["value"]))
            dec = max(decimals(r) for r in grp["value_raw"])
            for parent, kids in children.items():
                if parent not in vals or not all(k in vals for k in kids):
                    continue
                diff = abs(vals[parent] - sum(vals[k] for k in kids))
                checked += 1
                exact += diff == 0
                near += diff <= 0.5 * len(kids) * 10 ** -dec  # each published child may be rounded by half a unit
                worst = max(worst, diff)
    return {"checked": checked, "exact": exact, "within_rounding": near, "max_abs_diff": worst}


rows = []
for key, (table, df) in loaded.items():
    sc = None if is_ratio(df) else sum_check(df, table.dim_names, hier_by_id.get(key[0], {}))
    results[key]["sum_check"] = sc
    rows.append({"표": key[0], "파일": key[1], **(sc or {"checked": "비율 표(해당 없음)"})})
show(rows)

## 단위 4. 시점

**알고 싶은 것:** 주기, 범위, 빠진 시점, 표기 형식이 섞였는지

**방법**
- `count`, `first`, `last`: 시점 수와 범위
- `shapes`: 숫자를 9로 바꾼 표기 모양. 예: 2024 → 9999, 202401 → 999999
- 메타 `period_type`(Y/M/Q/H)도 함께 본다.

**읽는 법**
- 한 파일의 `shapes`가 2개 이상이면 연·월이 섞인 것이다. 추이를 같은 주기끼리만 계산하는 R3가 꼭 필요하다.
- 최근 10시점을 받았으니 `count`가 10보다 작으면 그 표의 시점이 원래 적거나 빠진 시점이 있는 것이다.
- `count ≥ 2`여야 추이 질문이 가능하다.

**docs/eda.md에 쓸 것:** 표별 주기, 범위, 시점 수, 표기 모양.

In [ ]:
# Unit 4: period range and notation shapes.
rows = []
for key, (table, df) in loaded.items():
    periods = sorted(set(df["period"]))
    shapes = sorted({re.sub(r"\d", "9", p) for p in periods})
    results[key]["period"] = {"count": len(periods), "first": periods[0] if periods else None,
                              "last": periods[-1] if periods else None, "shapes": shapes}
    rows.append({"표": key[0], "파일": key[1], "주기": table.meta.period_type, "시점 수": len(periods),
                 "범위": f"{periods[0]}~{periods[-1]}" if periods else "-", "표기 모양": ", ".join(shapes)})
show(rows)

## 단위 5. 단위·측정 종류

**알고 싶은 것:** 항목별 단위, 한 표 안 단위 혼재, 비율인지 개수인지, 단위 빈칸, 소수 자릿수

**방법**
- `distinct`: 단위별 셀 수, `items_with_mixed_units`: 단위가 2개 이상인 항목 수
- `blank_unit_cells`: 단위가 빈 셀 수, `meta_unit`: 공유 메타의 단위
- `measure_kind_guess`: 단위 이름에 %, 율, 당, ‰가 있으면 ratio로 추정(사람이 확인)
- `max_decimals_by_item`: 항목별 최대 소수 자릿수

**읽는 법**
- 한 표에 단위가 여러 개면(예: DT_1B8000G의 명·건) 항목끼리 합산하거나 비교하면 안 된다(R5).
- ratio 표는 합산과 구성비 계산이 금지다(R8). DT_1B26006_A01로 확인한다.
- `blank_unit_cells > 0`이면 답변에 단위를 못 붙인다. 메타 단위로 보충할지, 거절할지 결정이 필요하다. DT_1B81A24로 확인한다.
- 소수 자릿수는 골든셋 정답의 허용 오차 기준이 된다(T1-03).
- stock(인구 수)·flow(출생 건수)·index 구분은 자동으로 못 한다. 표 이름과 항목을 보고 직접 적는다(T1-07).

**docs/eda.md에 쓸 것:** 표별 단위 목록, 혼재 여부, 빈칸 수, 측정 종류(사람 확인), 소수 자릿수.

In [ ]:
# Unit 5: units, mixed units per item, blank units, ratio guess, decimals.
rows = []
for key, (table, df) in loaded.items():
    units = Counter(df["unit"])
    unit = {"distinct": dict(units),
            "items_with_mixed_units": int((df.groupby("item")["unit"].nunique() > 1).sum()),
            "blank_unit_cells": int(units.get("", 0)),
            "meta_unit": meta_by_id.get(key[0], {}).get("unit"),
            "measure_kind_guess": "ratio" if is_ratio(df) else "count-like (check)",
            "max_decimals_by_item": {k: int(v) for k, v in
                                     df.groupby("item")["value_raw"].agg(lambda s: max(map(decimals, s))).items()}}
    results[key]["unit"] = unit
    rows.append({"표": key[0], "파일": key[1],
                 "단위(셀 수)": ", ".join(f"{u or '(빈칸)'} {n}" for u, n in units.items()),
                 "단위 섞인 항목": unit["items_with_mixed_units"], "빈칸": unit["blank_unit_cells"],
                 "메타 단위": unit["meta_unit"] or "(빈칸)", "추정": unit["measure_kind_guess"],
                 "최대 소수 자릿수": max(unit["max_decimals_by_item"].values(), default=0)})
show(rows)

## 단위 6. 값·결측 기호와 빠진 조합

둘 다 "답할 수 없는 경우"의 근거라서 한 단위로 묶었다.

**알고 싶은 것**
- 숫자로 못 읽는 값(`-`, `x`, `…`, 잠정 `p` 등)이 어떤 종류로 얼마나 있나?
- 항목 × 범주 × 시점 조합 중 아예 행이 없는 칸이 얼마나 되나?

**방법**
- `flags`: B0 `clean`이 붙인 `value_flag`(ok / missing / suppressed / provisional / unparsed)별 개수
- `parsed_share`: 숫자로 읽힌 비율, `unparsed_examples`: 못 읽은 원문 예시(최대 10개)
- `missing_combos`: `expected` = 항목 수 × 시점 수 × 분류별 범주 수의 곱, `present` = 실제 행 수, `missing_share` = 1 − present / expected

**읽는 법**
- `unparsed`가 있으면 전처리가 모르는 기호다. 예시를 R4 기호 목록에 추가한다.
- `provisional`이 있으면 답변에 "잠정치" 표기가 필요하다.
- `missing_share`가 크면 "표에 없는 조건"이 흔하다는 뜻이다. T1-08 거절 판정과 T1-03 불가 문항의 근거다.
- 단, 분류끼리 짝이 정해진 표(예: 시도 아래 시군구)는 곱으로 잡은 `expected`가 과대라서 `missing_share`가 높게 나온다. 결측이 아니라 구조 때문이니 구분해서 적는다.

**docs/eda.md에 쓸 것:** 표별 기호 종류와 개수, 파싱률, 빠진 조합 비율과 원인(진짜 결측 / 구조).

In [ ]:
# Unit 6: value flags from the B0 cleaner and missing period x category combinations.
rows = []
for key, (table, df) in loaded.items():
    flags = Counter(df["flag"])
    expected = df["item"].nunique() * df["period"].nunique()
    for d in table.dim_names:
        col = f"code:{d}" if f"code:{d}" in df else f"dim:{d}"
        expected *= max(1, df[col].nunique())
    share = round(1 - len(df) / expected, 4) if expected else None
    results[key]["values"] = {"flags": dict(flags),
                              "parsed_share": round(df["value"].notna().mean(), 4) if len(df) else None,
                              "unparsed_examples": sorted(set(df.loc[df["flag"] == "unparsed", "value_raw"]))[:10]}
    results[key]["missing_combos"] = {"expected": expected, "present": len(df), "missing_share": share}
    rows.append({"표": key[0], "파일": key[1], "flag별 개수": ", ".join(f"{k} {v}" for k, v in flags.items()),
                 "파싱률": f"{results[key]['values']['parsed_share']:.1%}" if len(df) else "-",
                 "못 읽은 예시": ", ".join(results[key]["values"]["unparsed_examples"]) or "없음",
                 "조합(실제/기대)": f"{len(df)}/{expected}",
                 "빠진 조합": f"{share:.1%}" if share is not None else "-"})
show(rows)

## 단위 7. 질문 가능성 매트릭스

**알고 싶은 것:** 표마다 실전 1 질문 5종이 가능한가?

**방법:** 앞 단위 결과로 자동 판정한다.

| 유형 | 판정 기준 |
| --- | --- |
| 단순 수치 | ok 값이 1개 이상 (단위 6) |
| 추이 | 시점 2개 이상 (단위 4) |
| 비교 | 맨 아래 범주가 2개 이상인 분류가 있음 (단위 3) |
| 구성비 | 비율 표가 아니고(단위 5), 상위 범주(분모)가 있음(단위 3) |
| 최대최소 | 맨 아래 범주가 3개 이상인 분류가 있음 (단위 3) |

**읽는 법**
- 5개 표를 합쳐 5종이 모두 어딘가에서 가능해야 골든셋을 유형별로 고르게 만들 수 있다.
- 아무 표에서도 안 되는 유형이 있으면 표를 바꾸거나 추가한다.
- 자동 판정이라 틀릴 수 있다. 예: 단위가 섞인 표에서 비교가 True로 나와도 실제로는 같은 단위끼리만 된다. 사람이 고치고 이유를 적는다.

**docs/eda.md에 쓸 것:** 표 × 유형 O/X 표와 사람이 고친 칸의 이유.

In [ ]:
# Unit 7: question-type feasibility from units 3-6 (heuristic, reviewed by a person).
rows = []
for key, (table, df) in loaded.items():
    r = results[key]
    h = r["hierarchy"]
    leaves = {d: v["codes_in_data"] - v["non_leaf_in_data"] for d, v in h.items()}
    q = {"단순 수치": r["values"]["flags"].get("ok", 0) > 0,
         "추이": r["period"]["count"] >= 2,
         "비교": any(n >= 2 for n in leaves.values()),
         "구성비": r["unit"]["measure_kind_guess"] != "ratio" and any(v["non_leaf_in_data"] > 0 for v in h.values()),
         "최대최소": any(n >= 3 for n in leaves.values())}
    r["questions"] = q
    rows.append({"표": key[0], "파일": key[1], **{k: "O" if v else "X" for k, v in q.items()}})
show(rows)
covered = {k for row in rows for k, v in row.items() if v == "O"}
print("\n아무 표에서도 안 되는 유형:", sorted({"단순 수치", "추이", "비교", "구성비", "최대최소"} - covered) or "없음")

## 단위 8. 저장과 표 간 비교

**이 셀이 하는 일:** 단위 1~7 결과를 `data/eda/profile.json`에 저장하고(커밋 안 함), `docs/eda.md`에 붙일 표 간 비교표를 출력한다.

- 앞 단위를 일부만 돌렸으면 빠진 칸은 `-`로 나온다.
- `docs/eda.md`에는 셀 값(실제 인구 수 등)을 쓰지 않는다. 구조, 개수, 비율, 출처(조사명·표명·표 ID·링크)만 쓴다.
- 마지막 절 "다음 태스크에 넘길 것": T1-02(표별 질문 유형), T1-05(로더가 놓친 필드, R2 계층 판정, R4 기호, R5 단위 혼재, R7 행 선택), T1-03(허용 오차, 잠정치, 불가 문항), T1-08(거절 사유).

In [ ]:
# Unit 8: save everything measured so far and print the cross-table comparison for docs/eda.md.
profiles = [{"tbl_id": k[0], "file": k[1], **v} for k, v in results.items()]
(OUT_DIR / "profile.json").write_text(
    json.dumps({"profiles": profiles, "load_failures": {f"{k[0]}/{k[1]}": v for k, v in failures.items()}},
               ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(f"{len(profiles)} profiles -> {(OUT_DIR / 'profile.json').relative_to(root)}; load failures: {len(failures)}\n")


def get(p, *path):
    for k in path:
        p = p.get(k) if isinstance(p, dict) else None
    return "-" if p is None else p


show([{
    "표": p["tbl_id"], "주기": p.get("period_type", "-"), "셀": get(p, "size", "cells"), "항목": get(p, "size", "items"),
    "분류": " × ".join(f"{d}({n})" for d, n in get(p, "size", "codes_per_dim").items()) if "size" in p else "-",
    "시점": f"{get(p, 'period', 'first')}~{get(p, 'period', 'last')} ({get(p, 'period', 'count')})",
    "단위": ", ".join(u or "(빈칸)" for u in get(p, "unit", "distinct")) if "unit" in p else "-",
    "결측 기호": ", ".join(f"{k} {v}" for k, v in get(p, "values", "flags").items() if k != "ok") or "없음" if "values" in p else "-",
    "빠진 조합": f"{p['missing_combos']['missing_share']:.1%}" if get(p, "missing_combos", "missing_share") != "-" else "-",
    "합계 일치(정확/반올림 내/검사)": ("비율 표(해당 없음)" if p.get("sum_check", "-") is None else
                                  f"{get(p, 'sum_check', 'exact')}/{get(p, 'sum_check', 'within_rounding')}/{get(p, 'sum_check', 'checked')}"),
    "토큰(대략)": get(p, "size", "approx_tokens"),
} for p in profiles])